## CPSC 4970 Module 3 Homework - Jonathan Braun

In [9]:
import pandas as pd
import numpy as np

from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.compose import TransformedTargetRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from math import sqrt

In [10]:
train_url = "https://raw.githubusercontent.com/cdavidshaffer/CPSC4970-AI/master/data/m3train.csv"
test_url = "https://raw.githubusercontent.com/cdavidshaffer/CPSC4970-AI/master/data/m3test.csv"

training_data = pd.read_csv(train_url)
testing_data = pd.read_csv(test_url)

display(training_data.head())
display(testing_data.head())

,A,B,C,D,E,F,G,H,I,J,K,L,M,N
0,0.003555,92.159562,6.658918,0.0,1.994330,35.960534,290.003375,20.973961,7.523349,2410.680571,33.715837,1965.274105,34.370406,220.134692
1,0.015362,0.000000,20.380324,0.0,1.738552,35.118264,350.939667,25.471824,15.046698,1970.894250,39.224961,1965.274105,63.081429,198.121223
2,0.015351,0.000000,20.380324,0.0,1.738552,39.296796,271.766966,25.471824,15.046698,1970.894250,39.224961,1945.121257,27.813803,318.278075
3,0.018208,0.000000,6.284173,0.0,1.697776,38.274040,203.714027,31.087615,22.570047,1808.010428,41.208245,1954.034064,20.290963,306.354113
4,0.038841,0.000000,6.284173,0.0,1.697776,39.088964,241.076425,31.087615,22.570047,1808.010428,41.208245,1965.274105,36.785997,332.036493


,A,B,C,D,E,F,G,H,I,J,K,L,M
0,0.042205,168.959198,6.284173,0.0,1.749673,40.582078,319.804335,15.893032,52.663442,1808.010428,40.54715,1965.274105,44.653922
1,0.027743,168.959198,6.284173,0.0,1.749673,37.459117,312.687688,16.321229,52.663442,1808.010428,40.54715,1965.274105,51.969711
2,0.277305,0.000000,28.538219,0.0,2.016572,36.288691,366.952123,17.012498,30.093396,2475.834100,40.54715,1965.274105,31.333664
3,0.196540,0.000000,28.538219,0.0,2.016572,32.662556,341.154277,15.909954,30.093396,2475.834100,40.54715,1962.006076,68.809830
4,1.482478,0.000000,28.538219,0.0,2.016572,27.198743,168.130791,12.919755,30.093396,2475.834100,40.54715,1735.274150,87.237337


In [11]:
training_data.info()

testing_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 306 entries, 0 to 305
Data columns (total 14 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   A       306 non-null    float64
 1   B       306 non-null    float64
 2   C       306 non-null    float64
 3   D       306 non-null    float64
 4   E       306 non-null    float64
 5   F       306 non-null    float64
 6   G       306 non-null    float64
 7   H       306 non-null    float64
 8   I       306 non-null    float64
 9   J       306 non-null    float64
 10  K       306 non-null    float64
 11  L       306 non-null    float64
 12  M       306 non-null    float64
 13  N       306 non-null    float64
dtypes: float64(14)
memory usage: 33.6 KB
<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 13 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   A       100 non-null    float64
 1   B       100 non-null    float64
 2   C       100 non-null    fl

In [12]:
training_data = training_data.dropna()

X = training_data.iloc[:, :-1]
y = training_data.iloc[:, -1]

display(X.head())
display(y.head())

,A,B,C,D,E,F,G,H,I,J,K,L,M
0,0.003555,92.159562,6.658918,0.0,1.994330,35.960534,290.003375,20.973961,7.523349,2410.680571,33.715837,1965.274105,34.370406
1,0.015362,0.000000,20.380324,0.0,1.738552,35.118264,350.939667,25.471824,15.046698,1970.894250,39.224961,1965.274105,63.081429
2,0.015351,0.000000,20.380324,0.0,1.738552,39.296796,271.766966,25.471824,15.046698,1970.894250,39.224961,1945.121257,27.813803
3,0.018208,0.000000,6.284173,0.0,1.697776,38.274040,203.714027,31.087615,22.570047,1808.010428,41.208245,1954.034064,20.290963
4,0.038841,0.000000,6.284173,0.0,1.697776,39.088964,241.076425,31.087615,22.570047,1808.010428,41.208245,1965.274105,36.785997


0    220.134692
1    198.121223
2    318.278075
3    306.354113
4    332.036493
Name: N, dtype: float64

The training data is labeled, with the final column used as the target variable. Rows with missing values are removed using `dropna()`. The testing data is not used during training because it does not include the target column.

## Cross-Validated Model

In [13]:
pipeline = Pipeline([
    ("poly", PolynomialFeatures(include_bias=False)),
    ("scale", StandardScaler()),
    ("regr", TransformedTargetRegressor(transformer=StandardScaler()))
])

param_grid = [
    {
        "poly__degree": [1, 2, 3, 4, 5, 6],
        "regr__regressor": [LinearRegression()]
    },
    {
        "poly__degree": [1, 2, 3, 4, 5, 6],
        "regr__regressor": [Ridge()],
        "regr__regressor__alpha": [0.001, 0.01, 0.1, 1, 10, 100]
    },
    {
        "poly__degree": [1, 2, 3, 4, 5, 6],
        "regr__regressor": [Lasso(max_iter=100000)],
        "regr__regressor__alpha": [0.001, 0.01, 0.1, 1, 10]
    }
]

model = GridSearchCV(
    pipeline,
    param_grid=param_grid,
    scoring="neg_root_mean_squared_error",
    cv=5,
    n_jobs=-1
)

model.fit(X, y);

In [14]:
cv_rmse = -model.best_score_

print("Best cross-validated RMSE:", cv_rmse)

Best cross-validated RMSE: 31.049957795568297


Based on the cross-validation results, the model's estimated prediction error is approximately plus or minus 31.05 units using Root Mean Squared Error. Therefore, all predictions are plus or minus 31.05.

In [15]:
print("Best hyperparameters found by GridSearchCV:")
print(model.best_params_)

Best hyperparameters found by GridSearchCV:
{'poly__degree': 6, 'regr__regressor': Lasso(max_iter=100000), 'regr__regressor__alpha': 0.1}


In [16]:
X_grading = pd.read_csv('https://raw.githubusercontent.com/cdavidshaffer/CPSC4970-AI/master/data/m3test.csv')
predicted = model.predict(X_grading)